# Primate Motor Cortex Dataset — Exploratory Analysis

**Purpose:** Validate that real intracortical spike recordings from the Flint & Slutzky
center-out reaching dataset (Zenodo record 11550255) exhibit the cosine directional
tuning behaviour described in Georgopoulos et al. (1982). This confirms the statistical
model used by `src/generate_data.py` to generate synthetic spike data is grounded in
real primate motor cortex activity, before scaling the electrode-count sweep experiment.

**Dataset file used:** `MonkeyM_CO_20090303.mat` — one center-out session, monkey M,
96-channel Utah array implant, Flint & Slutzky / Northwestern University.

**Source:** https://zenodo.org/records/11550255

**Author:** Gaurav Joseph Zachariah (26MTCSED001) — M.Tech CSE (Data Science), SHUATS


## 1. Setup — imports and file path

In [ ]:
# Core numerical + plotting libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

# Path to the raw .mat file. Adjust if your folder layout differs.
# Recommended project layout: this notebook lives in notebooks/,
# and the raw .mat file lives in data/raw/ one level up.
DATA_PATH = "MonkeyM_CO_20090303.mat"


## 2. Load the raw MATLAB file

This particular file loads directly with `scipy.io.loadmat` (it is an older MATLAB
format, not the newer v7.3/HDF5 format some large neuroscience datasets use).
`squeeze_me=True` removes redundant array dimensions, and `struct_as_record=False`
lets us access MATLAB struct fields with normal Python dot notation (e.g. `bdf.units`)
instead of clunky record-array indexing.

In [ ]:
data = loadmat(DATA_PATH, squeeze_me=True, struct_as_record=False)

# Confirm the top-level variable name inside the file (should be 'bdf')
top_level_keys = [k for k in data.keys() if not k.startswith('__')]
print("Top-level variables in file:", top_level_keys)

bdf = data['bdf']


## 3. Extract the two pieces of data we need

- `bdf.units` — an array of 103 neuron structs, each with:
  - `id`  = [electrode channel number, sorted unit number on that channel]
  - `ts`  = every timestamp (in seconds) at which that neuron fired a spike
- `bdf.trial_table` — a (154 trials x 10 columns) array describing every trial:
  trial start time, unused placeholder columns, **direction code (0-7)**,
  timing checkpoints (go-cue, movement start, etc.), and an **outcome code**
  (82 = success/'R', 70 = fail/'F', 65 = abort/'A', -1 = invalid/no direction assigned).

In [ ]:
units = bdf.units
trial_table = bdf.trial_table

# Named column indices for trial_table, for readability in the code below
COL_DIRECTION = 4   # direction code, 0-7 (or -1 if no direction was assigned)
COL_GO_CUE    = 5   # timestamp: go-cue / movement window start
COL_REWARD    = 8   # timestamp: movement window end
COL_OUTCOME   = 9   # outcome code: 82 = success, 70 = fail, 65 = abort, -1 = invalid

print(f"Loaded {len(units)} neurons and {trial_table.shape[0]} trials.")


## 4. Keep only clean, successful trials

We only want trials where (a) a real direction was assigned (code 0-7, not the -1
placeholder) and (b) the monkey completed the reach successfully (outcome 82).
Failed/aborted trials may not reflect the neuron's true directional preference, since
the monkey did not complete a clean reach.

In [ ]:
is_real_direction = trial_table[:, COL_DIRECTION] != -1
is_successful      = trial_table[:, COL_OUTCOME] == 82
clean_trials = trial_table[is_real_direction & is_successful]

print(f"Using {clean_trials.shape[0]} clean trials out of {trial_table.shape[0]} total "
      f"({trial_table.shape[0] - clean_trials.shape[0]} excluded as failed/aborted/invalid).")


## 5. Map direction codes (0-7) to real compass directions

The direction codes are arbitrary lab-assigned labels with no inherent compass meaning.
To find the true angle for each code, we look at the monkey's actual hand position
(`bdf.pos`) at the start and end of a few sample trials per code, and compute the
angle of that movement vector.

In [ ]:
pos = bdf.pos                  # columns: [time, x, y]
pos_time = pos[:, 0]
pos_xy   = pos[:, 1:3]

def get_reach_angle(direction_code, trials, n_samples=5):
    """Average reach angle (degrees, 0-360) for a given direction code,
    measured from hand displacement (end position - start position) across
    a handful of sample trials of that direction."""
    sample_trials = trials[trials[:, COL_DIRECTION] == direction_code][:n_samples]
    angles = []
    for row in sample_trials:
        start_t, end_t = row[COL_GO_CUE], row[COL_REWARD]
        if np.isnan(start_t) or np.isnan(end_t):
            continue
        window = (pos_time >= start_t) & (pos_time <= end_t)
        if not np.any(window):
            continue
        start_xy = pos_xy[window][0]
        end_xy   = pos_xy[window][-1]
        dx, dy = end_xy - start_xy
        angles.append(np.degrees(np.arctan2(dy, dx)) % 360)
    return np.mean(angles) if angles else np.nan

direction_angles_deg = {d: get_reach_angle(d, clean_trials) for d in range(8)}
for d, angle in direction_angles_deg.items():
    print(f"Direction code {d}: reach angle ~ {angle:.1f} deg")


## 6. Compute each neuron's firing rate per direction (the tuning curve)

For every neuron, and for every direction, we count how many spikes it fired during
the movement window (go-cue to reward) of every clean trial of that direction, and
convert that count to a rate in spikes-per-second. This gives each neuron an 8-value
"tuning curve" — one average firing rate per direction.

In [ ]:
def firing_rate_by_direction(spike_times, trials):
    """Return an array of 8 mean firing rates (spikes/sec), one per direction code 0-7."""
    rates = np.full(8, np.nan)
    for d in range(8):
        d_trials = trials[trials[:, COL_DIRECTION] == d]
        trial_rates = []
        for row in d_trials:
            start_t, end_t = row[COL_GO_CUE], row[COL_REWARD]
            if np.isnan(start_t) or np.isnan(end_t) or end_t <= start_t:
                continue
            n_spikes = np.sum((spike_times >= start_t) & (spike_times <= end_t))
            trial_rates.append(n_spikes / (end_t - start_t))
        if trial_rates:
            rates[d] = np.mean(trial_rates)
    return rates


## 7. Batch process: compute tuning curves for ALL 103 neurons

This is the "batch processing loop" — rather than computing one neuron's tuning curve
by hand, we loop the function above over every neuron in the recording and collect the
results into one table. This lets us measure, across the whole population, how common
and how strong directional tuning actually is in this session — a single neuron alone
can't tell us whether the cosine-tuning model is a general property of the data or a
one-off coincidence.

In [ ]:
n_neurons = len(units)
all_rates = np.full((n_neurons, 8), np.nan)   # rows = neurons, columns = directions 0-7
neuron_ids = []

for i, neuron in enumerate(units):
    all_rates[i] = firing_rate_by_direction(neuron.ts, clean_trials)
    neuron_ids.append(tuple(neuron.id))

print(f"Computed tuning curves for {n_neurons} neurons.")
print("Tuning curve matrix shape:", all_rates.shape)


## 8. Rank neurons by tuning strength

Not every neuron is strongly direction-selective — that is expected and biologically
normal. We quantify "tuning strength" per neuron as its **modulation depth**: the
difference between its highest and lowest firing rate across the 8 directions. A large
modulation depth means the neuron's firing rate swings widely depending on direction
(strong tuning); a small one means it fires at roughly the same rate regardless of
direction (weak/no tuning). We rank all 103 neurons by this measure and keep the
strongest few to showcase — these give the clearest, most convincing visual evidence
of cosine tuning.

In [ ]:
modulation_depth = np.nanmax(all_rates, axis=1) - np.nanmin(all_rates, axis=1)

# Indices of the top 6 most strongly-tuned neurons, strongest first
TOP_N = 6
top_indices = np.argsort(modulation_depth)[::-1][:TOP_N]

print(f"Top {TOP_N} most direction-tuned neurons (by modulation depth, spikes/sec):")
for rank, idx in enumerate(top_indices, start=1):
    print(f"  #{rank}: neuron id {neuron_ids[idx]}  -  modulation depth = {modulation_depth[idx]:.1f} Hz")


## 9. Plot tuning curves for the top neurons on compass-labelled polar plots

Each subplot shows one neuron's firing rate at every compass direction, plotted around
a circle. A neuron with genuine cosine tuning should show one clear "lobe" pointing
toward its preferred direction, tapering off toward the opposite side.

In [ ]:
compass_labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
# Standard math angle (radians) for each direction code 0-7, in the order matching
# the compass mapping found in Section 5 (code 0 = N, going clockwise)
angles_rad = np.radians([90, 45, 0, 315, 270, 225, 180, 135])

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': 'polar'})

for ax, idx in zip(axes.flat, top_indices):
    rates = all_rates[idx]
    # Close the loop so the plotted line connects back to its starting point
    plot_angles = np.append(angles_rad, angles_rad[0])
    plot_rates  = np.append(rates, rates[0])

    ax.plot(plot_angles, plot_rates, marker='o', linewidth=2)
    ax.fill(plot_angles, plot_rates, alpha=0.15)
    ax.set_xticks(angles_rad)
    ax.set_xticklabels(compass_labels)
    ax.set_title(f"Neuron {neuron_ids[idx]}\nmodulation depth = {modulation_depth[idx]:.1f} Hz",
                 fontsize=10)

fig.suptitle("Directional Tuning Curves — Top 6 Most Strongly-Tuned Neurons\n"
             "(MonkeyM, center-out task, session 2009-03-03)", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/primate_tuning_curves_top6.png", dpi=150, bbox_inches='tight')
plt.show()


## 10. Save the summary results for reference

Save the full tuning-curve table (all 103 neurons x 8 directions) and the modulation
depth ranking to CSV, so these numbers can be reused later (e.g. as reference values
when calibrating `src/generate_data.py`) without re-running this notebook.

In [ ]:
import csv
import os

os.makedirs("../results/tables", exist_ok=True)

with open("../results/tables/primate_tuning_curves.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["channel", "unit", "modulation_depth_hz"] +
                     [f"rate_dir_{d}_{compass_labels[d]}_hz" for d in range(8)])
    for i in range(n_neurons):
        ch, unit = neuron_ids[i]
        writer.writerow([ch, unit, round(modulation_depth[i], 2)] +
                         [round(r, 2) if not np.isnan(r) else "" for r in all_rates[i]])

print("Saved: ../results/tables/primate_tuning_curves.csv")
print("Saved: ../results/figures/primate_tuning_curves_top6.png")


## Summary

- Loaded a real center-out reaching session (103 neurons, 154 trials, 140 successful)
  from the Flint & Slutzky primate motor cortex dataset.
- Confirmed the 8 direction codes correspond to 8 evenly-spaced compass directions
  (~45 degrees apart), recovered from real hand-position data.
- Computed a directional firing-rate tuning curve for every one of the 103 neurons.
- The most strongly direction-tuned neurons show a clear single-lobe cosine-shaped
  tuning curve, consistent with Georgopoulos et al. (1982) — validating that the
  statistical model behind `src/generate_data.py`'s synthetic spike generation is
  grounded in real primate motor cortex behaviour.
